# Lab 7 — Transfer Learning Quantique

## Objectifs
- Implémenter une architecture hybride : CNN classique (gelé) + tête quantique
- Comparer avec une tête classique fully-connected
- Analyser la convergence et l'accuracy
- Étudier l'impact du bruit quantique sur les performances

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset
import pennylane as qml
from pennylane import qnode
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import time

In [ ]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

num_classes = 4
batch_size = 32
n_epochs = 50
lr = 0.01

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

## Partie A — Extracteur de features classique (ResNet18 gelé)

On utilise ResNet18 pré-entraîné sur ImageNet comme extracteur de features. Le backbone est entièrement gelé : seuls les paramètres de la tête de classification (quantique ou classique) seront entraînés.

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in resnet.parameters():
    param.requires_grad = False

feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
feature_extractor.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = CIFAR10(root="./data", train=False, download=True, transform=transform)

selected_classes = [0, 1, 2, 3]
train_idx = [i for i, (_, label) in enumerate(train_dataset) if label in selected_classes]
test_idx = [i for i, (_, label) in enumerate(test_dataset) if label in selected_classes]

train_subset = Subset(train_dataset, train_idx[:2000])
test_subset = Subset(test_dataset, test_idx[:400])

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

print(f"Classes CIFAR-10 sélectionnées : {[train_dataset.classes[i] for i in selected_classes]}")
print(f"Échantillons d'entraînement : {len(train_subset)}")
print(f"Échantillons de test : {len(test_subset)}")

In [ ]:
def extract_features(dataloader, model):
    """Extrait les features à partir du backbone gelé."""
    model.eval()
    all_features = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            features = model(images)
            features = features.squeeze()
            all_features.append(features)
            all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)

X_train, y_train = extract_features(train_loader, feature_extractor)
X_test, y_test = extract_features(test_loader, feature_extractor)

pca = nn.Linear(512, n_qubits)
nn.init.xavier_uniform_(pca.weight)

X_train_reduced = torch.relu(pca(X_train)).detach()
X_test_reduced = torch.relu(pca(X_test)).detach()

X_train_norm = X_train_reduced / (X_train_reduced.max(dim=0, keepdim=True).values + 1e-8)
X_test_norm = X_test_reduced / (X_test_reduced.max(dim=0, keepdim=True).values + 1e-8)

X_train_norm = X_train_norm.clamp(0, np.pi)
X_test_norm = X_test_norm.clamp(0, np.pi)

print(f"Features d'entraînement : {X_train_norm.shape}")
print(f"Features de test : {X_test_norm.shape}")
print(f"Dimension originale ResNet18 : {X_train.shape[1]} → Réduite à {n_qubits} qubits")

## Partie B — Tête quantique VQC

On construit un circuit quantique variationnel (VQC) comme tête de classification. L'encodage angulaire mappe les features dans l'espace des phases, suivies de couches d'intrication paramétrées.

In [ ]:
n_layers = 3

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes=weight_shapes)

print(f"Paramètres du circuit quantique : {n_layers * n_qubits}")
print(f"Nombre de qubits : {n_qubits}")
print(f"Nombre de couches : {n_layers}")
print(f"\nSchéma du circuit :")
print(qml.draw(quantum_circuit)(np.random.rand(n_qubits), np.random.rand(n_layers, n_qubits)))

In [ ]:
class HybridModel(nn.Module):
    def __init__(self, quantum_head, n_qubits, num_classes):
        super().__init__()
        self.quantum_head = quantum_head
        self.classifier = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        x = self.quantum_head(x)
        x = self.classifier(x)
        return x

model_quantum = HybridModel(qlayer, n_qubits, num_classes)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Modèle hybride créé")
print(f"Paramètres entraînables : {count_parameters(model_quantum)}")
print(f"\nArchitecture :")
print(f"  Backbone : ResNet18 (gelé, {sum(p.numel() for p in feature_extractor.parameters())} paramètres)")
print(f"  Réduction : 512 → {n_qubits} (linear)")
print(f"  Tête : VQC {n_qubits} qubits × {n_layers} couches")
print(f"  Classifieur : Linear({n_qubits}, {num_classes})")

## Partie C — Entraînement

On entraîne les deux modèles (quantique et classique) avec le même optimiseur Adam et le même learning rate pour une comparaison équitable.

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, epochs, lr):
    """Entraîne un modèle et retourne les métriques."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    test_accuracies = []
    train_accuracies = []

    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            train_preds = torch.argmax(model(X_train), dim=1)
            train_acc = accuracy_score(y_train.numpy(), train_preds.numpy())
            train_accuracies.append(train_acc)

            test_outputs = model(X_test)
            preds = torch.argmax(test_outputs, dim=1)
            acc = accuracy_score(y_test.numpy(), preds.numpy())
            test_accuracies.append(acc)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {loss.item():.4f} — Train Acc: {train_acc:.4f} — Test Acc: {acc:.4f}")

    elapsed = time.time() - start_time
    return train_losses, test_accuracies, train_accuracies, elapsed

quantum_losses, quantum_accs, quantum_train_accs, quantum_time = train_model(
    model_quantum, X_train_norm, y_train, X_test_norm, y_test, n_epochs, lr
)

print(f"\nTemps d'entraînement (quantique) : {quantum_time:.2f}s")
print(f"Accuracy finale (quantique) : test={quantum_accs[-1]:.4f}, train={quantum_train_accs[-1]:.4f}")

## Partie D — Tête classique (baseline)

On crée une tête classique équivalente en nombre de paramètres pour servir de baseline de comparaison.

In [ ]:
class ClassicalHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_classical = ClassicalHead(512, n_qubits, num_classes)

print(f"Modèle classique créé")
print(f"Paramètres entraînables : {count_parameters(model_classical)}")
print(f"\nComparaison des paramètres :")
print(f"  Quantique : {count_parameters(model_quantum)} paramètres")
print(f"  Classique : {count_parameters(model_classical)} paramètres")

In [ ]:
classical_losses, classical_accs, classical_train_accs, classical_time = train_model(
    model_classical, X_train, y_train, X_test, y_test, n_epochs, lr
)

print(f"\nTemps d'entraînement (classique) : {classical_time:.2f}s")
print(f"Accuracy finale (classique) : test={classical_accs[-1]:.4f}, train={classical_train_accs[-1]:.4f}")

## Partie E — Comparaison détaillée

On compare les deux approches sur plusieurs axes : convergence, accuracy, temps d'entraînement et nombre de paramètres.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(quantum_losses, label="Quantique", color="purple", linewidth=2)
axes[0].plot(classical_losses, label="Classique", color="blue", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Courbe de loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(quantum_accs, label="Quantique (test)", color="purple", linewidth=2)
axes[1].plot(classical_accs, label="Classique (test)", color="blue", linewidth=2)
axes[1].plot(quantum_train_accs, label="Quantique (train)", color="purple", linestyle='--', alpha=0.5)
axes[1].plot(classical_train_accs, label="Classique (train)", color="blue", linestyle='--', alpha=0.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Courbe d'accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

metrics = ["Accuracy", "Temps (s)", "Paramètres"]
quantum_vals = [quantum_accs[-1], quantum_time, count_parameters(model_quantum)]
classical_vals = [classical_accs[-1], classical_time, count_parameters(model_classical)]

x = np.arange(len(metrics))
width = 0.35
axes[2].bar(x - width/2, quantum_vals, width, label="Quantique", color="purple", alpha=0.8)
axes[2].bar(x + width/2, classical_vals, width, label="Classique", color="blue", alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics)
axes[2].set_title("Comparaison des métriques")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print(f"{'Métrique':<25} {'Quantique':<15} {'Classique':<15}")
print("=" * 60)
print(f"{'Accuracy test':<25} {quantum_accs[-1]:<15.4f} {classical_accs[-1]:<15.4f}")
print(f"{'Accuracy train':<25} {quantum_train_accs[-1]:<15.4f} {classical_train_accs[-1]:<15.4f}")
print(f"{'Overfitting (train-test)':<25} {quantum_train_accs[-1]-quantum_accs[-1]:<15.4f} {classical_train_accs[-1]-classical_accs[-1]:<15.4f}")
print(f"{'Temps (s)':<25} {quantum_time:<15.2f} {classical_time:<15.2f}")
print(f"{'Paramètres':<25} {count_parameters(model_quantum):<15} {count_parameters(model_classical):<15}")
print(f"{'Ratio temps (Q/C)':<25} {quantum_time/classical_time:<15.2f}")
print("=" * 60)

delta = classical_accs[-1] - quantum_accs[-1]
if delta > 0:
    print(f"\n→ Le modèle classique surpasse le quantique de {delta:.4f} en accuracy.")
    print(f"  Cependant, le modèle quantique utilise {count_parameters(model_quantum)} paramètres")
    print(f"  vs {count_parameters(model_classical)} pour le classique.")
else:
    print(f"\n→ Le modèle quantique surpasse le classique de {-delta:.4f} en accuracy")
    print(f"  avec seulement {count_parameters(model_quantum)} paramètres entraînables !")

## Analyse des résultats

### Résultats attendus

| Métrique | Quantique (VQC) | Classique (MLP) |
|----------|----------------|-----------------|
| Accuracy test | ~0.60-0.75 | ~0.65-0.80 |
| Temps d'entraînement | 5-30s | 1-5s |
| Paramètres | ~16-28 | ~2060 |

### Observations clés

1. **Architecture hybride** : Le backbone ResNet18 extrait des features de haut niveau qui sont ensuite classifiées par la tête quantique. Cette approche permet de bénéficier à la fois du transfer learning classique et de l'expressivité quantique.

2. **Efficacité des paramètres** : Le modèle quantique utilise considérablement moins de paramètres que le MLP classique, tout en atteignant des performances comparables.

3. **Coût computationnel** : La simulation du circuit quantique sur CPU est plus lente que l'évaluation du MLP classique. Sur un QPU réel, cela pourrait être compensé par le parallelisme quantique.

4. **Convergence** : Le circuit quantique peut converger plus lentement en raison du paysage de loss plus complexe (plateaux et barrières).

### Points de discussion
- L'encodage angulaire (AngleEmbedding) est-il optimal pour ces features ?
- Comment le nombre de qubits et de couches affecte-t-il les performances ?
- Quel est l'impact du bruit quantique simulé ?

## Pour aller plus loin

### Exercice 1 — Backbone alternatif
Remplacer ResNet18 par EfficientNet-B0 comme extracteur de features et comparer les résultats.

```python
efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
for param in efficientnet.parameters():
    param.requires_grad = False
# Utiliser efficientnet.features comme extracteur
```

### Exercice 2 — Impact du nombre de qubits
Tester avec 6 et 8 qubits. Analyser comment l'accuracy et le temps d'entraînement évoluent.

### Exercice 3 — Bruit quantique
Ajouter du bruit dépolarisant au simulateur et observer la dégradation progressive :

```python
noise_levels = [0.0, 0.01, 0.05, 0.1]
for p in noise_levels:
    dev_noisy = qml.device('default.mixed', wires=n_qubits)
    # Ajouter qml.depolarizing_channel aux circuits
```

### Exercice 4 — Encodage alternatif
Remplacer l'AngleEmbedding par un encodage IQP (Instantaneous Quantum Polynomial) ou un encodage en amplitude et comparer.

### Exercice 5 — Analyse du surapprentissage
Augmenter le nombre d'époques à 200 et analyser le surapprentissage. Ajouter de la régularisation (dropout, weight decay) et observer l'impact.